# Keskar et al. (ICLR 2017), Figure 3(a,c,d): F1, C1, C2
### From-scratch PyTorch experiment · Google Colab · one T4 GPU

**Protocol:** train SB and LB separately, save the best (by full-training-set CE) endpoint of each run, then *load the saved endpoints and compute all points afresh at plotting time*. For each coordinate (\alpha\in[-1,2]), evaluate

\[
\theta(\alpha)=(1-\alpha)\theta_{\mathrm{SB}}+\alpha\theta_{\mathrm{LB}}.
\]

Every alpha has **four actual full-dataset measurements**: train/test mean cross-entropy and train/test classification accuracy. The training loss is mean *per-example* softmax cross-entropy. Each plotted value is produced by the trained checkpoints, not copied from the paper and not supplied as a literal. No precomputed curves or outside reproduction repositories are used.

**PDF-only sources:** Keskar et al., *On Large-Batch Training for Deep Learning: Generalization Gap and Sharp Minima*, Sections 2.2 and 2.2.1 (pp. 4–5), Figure 3 and caption (p. 6), Appendix A (p. 12), Appendix B.1/B.3/B.4 (p. 12). Appendix B.4 says “C3” inside the **C2/C4** section; this notebook takes that as a typo and applies the described VGG configuration to C2. The code is written independently from the paper's verbal specification. **This is a reproduction attempt, not a guarantee of matching the exact published curves.**

> **Run instructions:** In Colab, choose **Runtime → Change runtime type → T4 GPU** and **Run all**. By default checkpoints go to Google Drive (authorization requested); change `CONFIG["use_google_drive"]` to `False` for local-only outputs. Training six models and evaluating 61 alpha positions over complete train/test sets is compute-intensive and may require multiple Colab sessions. Resume works at completed-epoch boundaries. Do not delete your Drive checkpoint directory between sessions.

## 1. What comes from the paper; what is an explicit implementation choice

| Topic | Paper specifies | Implemented here / qualification |
|---|---|---|
| Models | F1: MNIST, 784→(512×5)→10 with BN/ReLU; C1: CIFAR-10, two Conv(64,5, stride 2)–MaxPool(3) stages →384→192→10; C2: CIFAR-10, 13 conv layers in blocks [2×64,2×128,3×256,3×512,3×512] with MaxPool(2) after every block →512→10 | Exactly these layer counts; BN on hidden layers, not the final logits; CE uses **raw logits** rather than an explicit softmax + logarithm |
| Batch sizes | 256 (SB); 10% of train set (LB) | F1: 256/6000; C1 and C2: 256/5000 |
| Optimizer | Adam for both | PyTorch Adam defaults, LR 0.001, betas (0.9, 0.999), eps 1e-8, weight decay 0; **the paper does not state these hyperparameters** |
| Training | Until loss ceased improving; no epoch limit | Full-training-set CE monitored after each epoch; min improvement 1e-4, patience 15, safety cap 100 epochs; cap and thresholds are **not** in the paper |
| Figure 3 | α in [−1,2]; 0=SB and 1=LB; left axis CE, right axis accuracy; solid train/dashed test | 61 alpha grid locations plus inclusion of 0 and 1 (61 total with defaults); four full-dataset metrics per location; automatic loss axis bounds |
| Initialization | Five random trials, a random SB/LB pair chosen for Figure 3; it does **not** say whether a pair shares its start | **Shared initial model parameters per network**, separate fresh Adam state. Thus batch size is the controlled intervention; results remain one random trial and interpolation is one coordinate-dependent line, not a general proof about minima |
| BN at untrained α | Not specified | **Recalibrate** BN means/variances independently at every α on a *fixed 2048-example training-only subset* (no gradients, dropout off, cumulative per-batch averaging); evaluate full train and test sets afterward. Do not interpolate saved BN buffers. This estimates feature statistics at the interpolated weights, but differs from the checkpoints' training-time running stats and is approximate |
| Alternatives to BN recalibration | Not specified | (1) Interpolate endpoint running statistics: cheap but intermediate variances may not match activations; (2) reuse one endpoint's stats: biased; (3) recalculate stats: chosen, more computation and dependence on calibration protocol |
| Architectures' missing details | No precise padding, C1 pooling stride, BN location/order, BN eps/momentum, initialization distribution, or softmax numerical implementation | Conv→BN→ReLU; same-padding for stride-1 conv; C1 conv pad 2, pool(3,stride 2,pad 0); PyTorch module initialization/BN defaults; dropout p=0.5 *after* the C1 dense hidden activations and C2 512-unit hidden activation; final linear logits |
| Data preprocessing | Appendix A says MNIST/CIFAR used without preprocessing | Raw pixels converted to float in [0,1] only. No augmentation or per-channel standardization. The numeric scaling was not stated by the paper |
| T4 memory constraint | Does not discuss accumulation/BN microbatches | **Accumulate summed-example gradients** into exactly one Adam step per effective batch of 256/5000/6000; BN is computed per *physical microbatch*, not over the full effective batch. This is an explicit approximation, especially for LB, and is **not exactly equivalent** to original full-batch BN. The final undersized physical chunk is avoided by balanced splitting |
| Trial count, stopping, interpolation density, randomness | 5 runs summarized; only one randomly selected pair plotted | 1 seeded paired run per network, epoch-level resumable; configurable alpha density and BN subset. Endpoints cannot be assumed to match paper Table 2 exactly |

**Practical interpretation:** the plot shows the behavior of the *particular SB↔LB direction*, including extrapolation, under a fixed BN-recalibration convention. It does not estimate Hessian eigenvalues or the paper's separate sharpness metric. Using a common initialization removes one confound but does not make interpolation invariant to hidden-unit symmetries or weight reparameterization.

**Data leakage boundary:** training-set data may calibrate BN statistics, as in normal training; the test set is never used to fit weights, select checkpoints, or calibrate normalization. Checkpoint selection uses **training cross-entropy only**.

In [ ]:
# Cell 1 — Configuration: all experimentally chosen numbers live here, not in the plotting code.
from pathlib import Path

CONFIG = {
    "seed": 1729,
    "networks": ("F1", "C1", "C2"),
    "use_google_drive": True,       # For persistent checkpoints across Colab sessions.
    "drive_dir": "/content/drive/MyDrive/keskar_fig3_f1_c1_c2",
    "local_dir": "/content/keskar_fig3_f1_c1_c2",
    "data_dir": "/content/keskar_fig3_datasets",
    "num_workers": 1,             # Keeps Colab shared-memory use low for 5000-image batches.
    "pin_memory": True,
    "lr": 1e-3,
    "adam_betas": (0.9, 0.999),
    "adam_eps": 1e-8,
    "weight_decay": 0.0,
    "max_epochs": 100,              # Computational safety cap; NOT specified in paper.
    "patience": 15,                # Consecutive epochs without a significant CE drop.
    "min_delta": 1e-4,             # Plateau threshold in CE units.
    "save_every_epochs": 1,         # Resume progress in completed epochs.
    "bn_calibration_examples": 2048,# Fixed training subset; NOT specified in paper.
    "alpha_min": -1.0,
    "alpha_sb": 0.0,
    "alpha_lb": 1.0,
    "alpha_max": 2.0,
    "alpha_grid_points": 61,        # includes SB/LB endpoints with these bounds.
    "input_divisor": 255.0,        # uint8 pixels -> float32 in [0,1].
    "test_run_synthetic_checks": True,
    "models": {
        "F1": {"dataset": "MNIST",    "small_batch": 256, "large_batch": 6000,
               "physical_batch": 256, "eval_batch": 1024, "calib_batch": 256},
        "C1": {"dataset": "CIFAR10", "small_batch": 256, "large_batch": 5000,
               "physical_batch": 128, "eval_batch": 256,  "calib_batch": 128},
        "C2": {"dataset": "CIFAR10", "small_batch": 256, "large_batch": 5000,
               "physical_batch": 32,  "eval_batch": 64,   "calib_batch": 32},
    },
}

PLOT_STYLE = {             # Plot layout/format configuration; NO fixed metric data.
    "rows": 1,
    "figsize": (18, 5),
    "dpi": 180,
    "line_width": 1.7,
    "train_ls": "-",
    "test_ls": "--",
    "loss_color": "#225bb3",
    "accuracy_color": "#ca3030",
    "accuracy_ylim": (0, 100),
    "grid_alpha": 0.35,
    "file_name": "figure3_F1_C1_C2_computed.png",
}

In [ ]:
# Cell 2 — Imports, GPU environment, persistent output directory, seeds.
import math, json, hashlib, random, time, copy, gc, csv
from collections import OrderedDict

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision.datasets import MNIST, CIFAR10
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

if CONFIG["use_google_drive"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = Path(CONFIG["drive_dir"])
    except ImportError:
        print("Outside Colab; using local directory instead of Google Drive.")
        OUT_DIR = Path(CONFIG["local_dir"])
else:
    OUT_DIR = Path(CONFIG["local_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path(CONFIG["data_dir"])
DATA_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))
else:
    print("WARNING: no CUDA GPU detected. Full training is impractical on CPU.")

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
# FP32 throughout: no automatic mixed precision and no TF32 approximation.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False

print("PyTorch:", torch.__version__, " | Artifacts:", OUT_DIR)

## 2. Datasets and architecture definitions

The dataset objects below use the *original uint8 images* supplied by `torchvision`, avoiding expensive repeated PIL conversions. The only pixel transformation is an on-device division by 255. **No outside implementation of the paper is imported.** The last classification layer emits logits because `torch.nn.functional.cross_entropy` already evaluates log-softmax stably.

In [ ]:
# Cell 3 — Define F1/C1/C2 directly from the PDF's layer descriptions.
class F1Network(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        in_features = 784
        for _ in range(5):
            layers += [nn.Linear(in_features, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True)]
            in_features = 512
        self.hidden = nn.Sequential(*layers)
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        return self.classifier(self.hidden(x))


class C1Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        # For 32x32 input with the *specified/assumed* strides: [64,1,1].
        self.hidden = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 384), nn.BatchNorm1d(384), nn.ReLU(inplace=True), nn.Dropout(p=0.5),
            nn.Linear(384, 192), nn.BatchNorm1d(192), nn.ReLU(inplace=True), nn.Dropout(p=0.5),
        )
        self.classifier = nn.Linear(192, 10)

    def forward(self, x):
        return self.classifier(self.hidden(self.features(x)))


class C2Network(nn.Module):
    def __init__(self):
        super().__init__()
        # The PDF calls for 2,2,3,3,3 stride-1 3x3 convs, 2x2 pool after each block.
        blocks = []
        in_channels = 3
        for out_channels, depth in [(64, 2), (128, 2), (256, 3), (512, 3), (512, 3)]:
            for _ in range(depth):
                blocks += [
                    nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
                    nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
                ]
                in_channels = out_channels
            blocks.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.features = nn.Sequential(*blocks)
        # 32x32 -> 1x1 after five max pools; 512-channel flat feature.
        self.hidden = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Dropout(p=0.5),
        )
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        return self.classifier(self.hidden(self.features(x)))


def new_model(name):
    return {"F1": F1Network, "C1": C1Network, "C2": C2Network}[name]()


def get_dataset(dataset_name):
    '''Return (train, test) as TensorDatasets, retaining original uint8 bytes.'''
    if dataset_name == "MNIST":
        train_raw = MNIST(root=DATA_DIR, train=True, download=True)
        test_raw = MNIST(root=DATA_DIR, train=False, download=True)
        def make(raw):
            return TensorDataset(raw.data.unsqueeze(1).contiguous(),
                                 torch.as_tensor(raw.targets, dtype=torch.long))
    elif dataset_name == "CIFAR10":
        train_raw = CIFAR10(root=DATA_DIR, train=True, download=True)
        test_raw = CIFAR10(root=DATA_DIR, train=False, download=True)
        def make(raw):
            # CIFAR-10 data is NHWC uint8; convert to NCHW without normalization.
            x = torch.from_numpy(raw.data).permute(0, 3, 1, 2).contiguous()
            return TensorDataset(x, torch.as_tensor(raw.targets, dtype=torch.long))
    else:
        raise ValueError(dataset_name)
    return make(train_raw), make(test_raw)

DATASETS = {}  # Lazy cache; CIFAR reused by C1 and C2.

def datasets_for(name):
    key = CONFIG["models"][name]["dataset"]
    if key not in DATASETS:
        DATASETS[key] = get_dataset(key)
    return DATASETS[key]


def loader_for(dataset, batch_size, *, shuffle=False, generator=None):
    num_workers = CONFIG["num_workers"]
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, generator=generator,
        num_workers=num_workers,
        pin_memory=(CONFIG["pin_memory"] and DEVICE.type == "cuda"),
        persistent_workers=(num_workers > 0),
        **({"prefetch_factor": 1} if num_workers > 0 else {}),
    )


def to_device_pixels(x_uint8, targets):
    x = x_uint8.to(DEVICE, non_blocking=True).to(torch.float32)
    x.div_(CONFIG["input_divisor"])
    y = targets.to(DEVICE, non_blocking=True)
    return x, y

## 3. Actual training and checkpointing

**Effective batches and physical microbatches are distinct.** A DataLoader gives an effective batch (e.g. 5,000 samples), which is divided into approximately equal physical chunks of at most the specified microbatch size. Each chunk's **sum** of sample CE is divided by the original effective-batch count, accumulated by backpropagation, and *one* Adam step is taken. This preserves the sample-weighted gradient except for differences caused by BN's microbatch statistics. The balanced split avoids a tiny final microbatch.

A `*_best.pt` contains the trained model parameters and BN buffers at the lowest **full-training-set** CE seen within the stopping rule. A `*_progress.pt` also contains the Adam state and RNG states to resume safely at an epoch boundary. An optimizer is **never reused** between SB and LB. Neither the test set nor the interpolation curves enter checkpoint selection.

In [ ]:
# Cell 4 — Actual full-dataset measurements, training, and resumable checkpoint manager.
def model_initial_seed(name):
    # Independent starts BETWEEN model families; IDENTICAL start WITHIN SB/LB pair.
    return CONFIG["seed"] + {"F1": 0, "C1": 10000, "C2": 20000}[name]


def initialize_identically(name):
    seed = model_initial_seed(name)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return new_model(name).to(DEVICE)


def initial_digest(model):
    sha = hashlib.sha256()
    for key, param in model.named_parameters():
        sha.update(key.encode("utf-8"))
        sha.update(param.detach().cpu().contiguous().numpy().tobytes())
    return sha.hexdigest()


def training_signature(name, regime):
    # Excludes max_epochs so a capped run can be extended across Colab sessions.
    c = CONFIG["models"][name]
    b = c["small_batch"] if regime == "SB" else c["large_batch"]
    setting = {"name": name, "regime": regime, "effective_batch": b,
               "physical_batch": c["physical_batch"],
               "model_initial_seed": model_initial_seed(name),
               "lr": CONFIG["lr"], "betas": CONFIG["adam_betas"],
               "eps": CONFIG["adam_eps"], "weight_decay": CONFIG["weight_decay"],
               "patience": CONFIG["patience"], "min_delta": CONFIG["min_delta"],
               "input_divisor": CONFIG["input_divisor"]}
    return hashlib.sha256(json.dumps(setting, sort_keys=True).encode()).hexdigest()


def ckpt_path(name, regime, purpose):
    return OUT_DIR / f"{name}_{regime}_{purpose}.pt"


def save_atomic(payload, path):
    temp_path = path.with_suffix(path.suffix + ".partial")
    torch.save(payload, temp_path)
    temp_path.replace(path)


def torch_load_own(path):
    # These are locally generated checkpoints containing tensors and primitive metadata.
    return torch.load(path, map_location="cpu", weights_only=True)


@torch.inference_mode()
def measure_dataset(model, data_loader):
    '''Exact sample-weighted mean CE and top-1 accuracy on the entire supplied loader.'''
    model.eval()
    summed_loss, summed_correct, count = 0.0, 0, 0
    for raw, y_cpu in data_loader:
        x, y = to_device_pixels(raw, y_cpu)
        logits = model(x)
        summed_loss += F.cross_entropy(logits, y, reduction="sum").item()
        summed_correct += int((logits.argmax(dim=1) == y).sum().item())
        count += y.numel()
    assert count == len(data_loader.dataset)
    return {"ce": summed_loss / count, "accuracy": 100.0 * summed_correct / count,
            "examples": count}


def train_one(name, regime):
    config_model = CONFIG["models"][name]
    train_data, _ = datasets_for(name)
    effective_batch = (config_model["small_batch"] if regime == "SB"
                       else config_model["large_batch"])
    physical_batch = config_model["physical_batch"]
    signature = training_signature(name, regime)
    best_file = ckpt_path(name, regime, "best")
    progress_file = ckpt_path(name, regime, "progress")

    model = initialize_identically(name)
    starting_digest = initial_digest(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"],
                                 betas=CONFIG["adam_betas"], eps=CONFIG["adam_eps"],
                                 weight_decay=CONFIG["weight_decay"])
    first_epoch, best_loss, stale = 1, float("inf"), 0
    if progress_file.exists():
        progress = torch_load_own(progress_file)
        if progress["signature"] != signature or progress["initial_digest"] != starting_digest:
            raise RuntimeError(f"Stored progress settings/initialization mismatch: {progress_file}")
        model.load_state_dict(progress["model_state"])
        optimizer.load_state_dict(progress["optimizer_state"])
        # Loading to GPU model sometimes leaves optimizer buffers on CPU; move them explicitly.
        for state in optimizer.state.values():
            for key, value in state.items():
                if isinstance(value, torch.Tensor):
                    state[key] = value.to(DEVICE)
        first_epoch = progress["epoch"] + 1
        best_loss, stale = progress["best_loss"], progress["stale"]
        torch.set_rng_state(progress["cpu_rng"])
        if DEVICE.type == "cuda" and progress.get("cuda_rng") is not None:
            torch.cuda.set_rng_state(progress["cuda_rng"], device=DEVICE)
        if progress.get("stop_reason") == "plateau":
            if not best_file.exists():
                raise FileNotFoundError("Progress says complete but best weights are missing.")
            print(f"{name} {regime}: plateau reached at epoch {progress['epoch']}; using {best_file.name}")
            del model, optimizer
            return best_file
        print(f"{name} {regime}: resuming at epoch {first_epoch}")
    if first_epoch > CONFIG["max_epochs"]:
        if not best_file.exists():
            raise FileNotFoundError(f"Missing best checkpoint: {best_file}")
        print(f"{name} {regime}: reached epoch cap; using {best_file.name}")
        del model, optimizer
        return best_file

    eval_train_loader = loader_for(train_data, config_model["eval_batch"])
    for epoch in range(first_epoch, CONFIG["max_epochs"] + 1):
        tic = time.perf_counter()
        # Epoch-dependent deterministic shuffle, independent of DataLoader worker scheduling.
        shuffle_seed = model_initial_seed(name) + (0 if regime == "SB" else 1000000) + epoch
        epoch_gen = torch.Generator().manual_seed(shuffle_seed)
        train_loader = loader_for(train_data, effective_batch, shuffle=True, generator=epoch_gen)
        model.train()
        epoch_updates = 0
        seen = 0
        for raw_batch, y_batch in train_loader:
            batch_count = len(y_batch)
            # tensor_split makes nearly equal chunks: avoids a BN chunk of size ~1.
            n_chunks = math.ceil(batch_count / physical_batch)
            raw_parts = torch.tensor_split(raw_batch, n_chunks)
            y_parts = torch.tensor_split(y_batch, n_chunks)
            optimizer.zero_grad(set_to_none=True)
            for x_raw, y_raw in zip(raw_parts, y_parts):
                x, y = to_device_pixels(x_raw, y_raw)
                logits = model(x)
                (F.cross_entropy(logits, y, reduction="sum") / batch_count).backward()
            optimizer.step()
            epoch_updates += 1
            seen += batch_count
        assert seen == len(train_data)
        del train_loader
        exact_train = measure_dataset(model, eval_train_loader)
        epoch_loss = exact_train["ce"]
        if not math.isfinite(epoch_loss):
            raise FloatingPointError(f"Nonfinite training CE at {name} {regime}, epoch {epoch}.")
        improvement = (best_loss - epoch_loss) > CONFIG["min_delta"]
        if improvement:
            best_loss, stale = epoch_loss, 0
            save_atomic({"model_state": {k: v.detach().cpu().clone()
                                          for k, v in model.state_dict().items()},
                         "network": name, "regime": regime, "epoch": epoch,
                         "training_ce": epoch_loss, "training_accuracy": exact_train["accuracy"],
                         "signature": signature, "initial_digest": starting_digest}, best_file)
        else:
            stale += 1
        stop_reason = ("plateau" if stale >= CONFIG["patience"]
                       else "epoch_cap" if epoch >= CONFIG["max_epochs"] else None)
        if epoch % CONFIG["save_every_epochs"] == 0 or stop_reason is not None:
            save_atomic({"model_state": {k: v.detach().cpu().clone()
                                          for k, v in model.state_dict().items()},
                         "optimizer_state": optimizer.state_dict(),
                         "signature": signature, "initial_digest": starting_digest,
                         "epoch": epoch, "best_loss": best_loss, "stale": stale,
                         "stop_reason": stop_reason,
                         "cpu_rng": torch.get_rng_state(),
                         "cuda_rng": torch.cuda.get_rng_state(DEVICE)
                                     if DEVICE.type == "cuda" else None}, progress_file)
        print(f"{name:2s} {regime} | epoch {epoch:3d}/{CONFIG['max_epochs']} "
              f"| {epoch_updates:3d} optimizer steps | train CE {epoch_loss:.5f} "
              f"| train acc {exact_train['accuracy']:.2f}% "
              f"| best CE {best_loss:.5f} | plateau {stale}/{CONFIG['patience']} "
              f"| {time.perf_counter()-tic:.1f}s", flush=True)
        if stop_reason is not None:
            print(f"{name} {regime}: stopped due to {stop_reason}")
            break
    if not best_file.exists():
        raise FileNotFoundError("Training ended before any best model was saved.")
    del model, optimizer, eval_train_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best_file

In [ ]:
# Cell 5 — Cheap synthetic consistency checks (do not affect the plotted results).
if CONFIG["test_run_synthetic_checks"]:
    for model_name in CONFIG["networks"]:
        mm = initialize_identically(model_name)
        n_params = sum(p.numel() for p in mm.parameters())
        shape = (2, 1, 28, 28) if model_name == "F1" else (2, 3, 32, 32)
        mm.eval()
        with torch.inference_mode():
            output = mm(torch.randn(shape, device=DEVICE))
        assert tuple(output.shape) == (2, 10)
        assert torch.isfinite(output).all()
        print(f"{model_name}: output {tuple(output.shape)}, parameters {n_params:,}")
        del mm, output
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 4. Train all six endpoints, or resume automatically

The next cell can take **many hours** on a single T4. Rerunning it detects `*_progress.pt` and continues at the next epoch, or uses an already-completed checkpoint. This is the actual experiment: **no example/pretrained weights are ever loaded**. Both training regimes start from the same deterministic initialization for each architecture.

If Colab interrupts a run, remount the same Drive and rerun the notebook. A 100-epoch limit is only a safety cap: increase `CONFIG["max_epochs"]` and rerun if the training CE is still improving. For interpretability, check training progress and checkpoint endpoint performance before interpreting the curves.

In [ ]:
# Cell 6 — FULL TRAINING (six runs). Run this before constructing the Figure 3 curves.
checkpoint_files = {}
for name in CONFIG["networks"]:
    for regime in ("SB", "LB"):
        checkpoint_files[(name, regime)] = train_one(name, regime)
        print("Saved best endpoint:", checkpoint_files[(name, regime)])
assert len(checkpoint_files) == 2 * len(CONFIG["networks"])

## 5. Batch-normalization calibration and interpolation

At every α: (1) load the **two saved trained states** (once for that panel), (2) interpolate **only trainable parameters** directly, (3) reset all BN running means, variances, and tracked counts, (4) run a fixed, seeded training-only subset through the interpolated model with **BN in training mode, dropout in evaluation mode and gradients disabled**; and (5) evaluate the full train/test splits with both BN and dropout in evaluation mode. The cumulative BN setting (`momentum=None`) gives equal weight to calibration microbatches; default calibration sizes divide 2048 exactly, avoiding partial-batch bias. The fixed subset makes the whole curve internally consistent.

**BN caveats:** BN running variances are not parameters of the neural optimization problem, and there is no unique “correct” BN evaluation convention. This implementation measures CE/accuracy after recalibration. At α=0 or α=1, it may differ from evaluating the corresponding raw checkpoint with its original, training-time BN buffers; an endpoint comparison below reports this difference. In particular, all four curves are evaluated under the **same recalibration policy** rather than mixing different endpoint and intermediate definitions.

In [ ]:
# Cell 7 — Load saved endpoints, interpolate trained weights, recalibrate BN, measure ALL samples.
def fixed_calibration_subset(train_dataset, network_name):
    n = min(CONFIG["bn_calibration_examples"], len(train_dataset))
    gen = torch.Generator().manual_seed(CONFIG["seed"] + 333333)
    indices = torch.randperm(len(train_dataset), generator=gen)[:n].tolist()
    return Subset(train_dataset, indices)


@torch.no_grad()
def interpolate_parameters_(model, small_params, large_params, alpha):
    '''Apply theta(alpha) = (1-alpha)*small + alpha*large to TRAINABLE parameters.

    States are on the current device; BN running buffers are purposely excluded.
    torch.lerp also supports extrapolation (alpha outside [0,1]).
    '''
    for key, param in model.named_parameters():
        param.copy_(torch.lerp(small_params[key], large_params[key], float(alpha)))


@torch.no_grad()
def recalibrate_batchnorm(model, calibration_loader):
    # IMPORTANT: model.eval() disables dropout; only the BN modules are reenabled.
    model.eval()
    bn_modules = [module for module in model.modules()
                  if isinstance(module, nn.modules.batchnorm._BatchNorm)]
    original_momenta = [module.momentum for module in bn_modules]
    for module in bn_modules:
        assert module.track_running_stats, "This protocol requires BN running statistics."
        module.reset_running_stats()
        module.momentum = None  # Cumulative (equal-batch-weighted) running statistics.
        module.train()          # Activates updates on calibration images, but NO dropout.
    for images_raw, target in calibration_loader:
        x, _ = to_device_pixels(images_raw, target)
        model(x)                # No gradient; do not use labels for calibration.
    model.eval()
    for module, momentum in zip(bn_modules, original_momenta):
        module.momentum = momentum


# All buffer keys are derived from module structure, not a hardcoded key-pattern guess.
def get_bn_buffer_keys(model):
    buffer_keys = set()
    for module_name, module in model.named_modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            for key, _value in module.named_buffers(recurse=False):
                buffer_keys.add(f"{module_name}.{key}" if module_name else key)
    return buffer_keys

def read_and_validate_endpoints(name, model):
    small_file = ckpt_path(name, "SB", "best")
    large_file = ckpt_path(name, "LB", "best")
    if not small_file.exists() or not large_file.exists():
        raise FileNotFoundError(f"Train BOTH {name} endpoints first: {small_file}, {large_file}")
    s, l = torch_load_own(small_file), torch_load_own(large_file)
    assert s["network"] == l["network"] == name
    assert s["regime"] == "SB" and l["regime"] == "LB"
    assert s["signature"] == training_signature(name, "SB")
    assert l["signature"] == training_signature(name, "LB")
    assert s["initial_digest"] == l["initial_digest"], (
        "SB and LB must begin from identical initialization!")
    keys = set(dict(model.named_parameters()))
    assert keys == set(s["model_state"]) - get_bn_buffer_keys(model)
    assert keys == set(l["model_state"]) - get_bn_buffer_keys(model)
    return s, l


def compute_curve_directly_from_checkpoints(name):
    '''Called by the plotting cell: every point is evaluated now, not read from CSV.'''
    model = new_model(name).to(DEVICE)
    sb, lb = read_and_validate_endpoints(name, model)
    train_data, test_data = datasets_for(name)
    cfg = CONFIG["models"][name]
    train_loader = loader_for(train_data, cfg["eval_batch"])
    test_loader = loader_for(test_data, cfg["eval_batch"])
    cal_subset = fixed_calibration_subset(train_data, name)
    cal_loader = loader_for(cal_subset, cfg["calib_batch"])

    param_keys = set(dict(model.named_parameters()))
    small_params = {k: sb["model_state"][k].to(DEVICE) for k in param_keys}
    large_params = {k: lb["model_state"][k].to(DEVICE) for k in param_keys}

    # Checkpoint metric diagnostics with ORIGINAL endpoint BN buffers; not used for plotting.
    original_endpoint_metrics = {}
    for regime, checkpoint in (("SB", sb), ("LB", lb)):
        model.load_state_dict(checkpoint["model_state"])
        original_endpoint_metrics[regime] = {
            "train": measure_dataset(model, train_loader),
            "test": measure_dataset(model, test_loader),
        }
    # The grid automatically includes 0 and 1 even if an alternative density is chosen.
    alphas = np.unique(np.concatenate([
        np.linspace(CONFIG["alpha_min"], CONFIG["alpha_max"], CONFIG["alpha_grid_points"]),
        np.asarray([CONFIG["alpha_sb"], CONFIG["alpha_lb"]], dtype=np.float64),
    ]))
    result = []
    print(f"{name}: recalibrating BN and measuring complete datasets at {len(alphas)} alpha values.")
    begin = time.perf_counter()
    for k, alpha in enumerate(alphas):
        with torch.no_grad():
            interpolate_parameters_(model, small_params, large_params, alpha)
        recalibrate_batchnorm(model, cal_loader)
        train_metrics = measure_dataset(model, train_loader)
        test_metrics = measure_dataset(model, test_loader)
        record = {"network": name, "alpha": float(alpha),
                  "train_ce": train_metrics["ce"], "test_ce": test_metrics["ce"],
                  "train_acc": train_metrics["accuracy"], "test_acc": test_metrics["accuracy"]}
        assert all(math.isfinite(record[key])
                   for key in ("train_ce", "test_ce", "train_acc", "test_acc"))
        result.append(record)
        print(f"{name} {k+1:3d}/{len(alphas)} alpha={alpha:+.3f} "
              f"train CE={record['train_ce']:.4f} test CE={record['test_ce']:.4f} "
              f"train acc={record['train_acc']:.2f}% "
              f"test acc={record['test_acc']:.2f}%", flush=True)
    print(f"{name}: curve computation completed in {(time.perf_counter()-begin)/60:.1f} minutes.")
    del model, small_params, large_params, train_loader, test_loader, cal_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result, original_endpoint_metrics

## 6. Figure 3(a,c,d): regenerate plots by evaluating the saved checkpoints

**This plotting cell recomputes every plotted coordinate from the checkpoint weights in the current run.** It does not load CSV values or manufacture reference curve arrays. The CSV/JSON and PNG are written *after* collecting fresh measurements so you can inspect and archive the experiment. Changing the alpha grid or BN calibration size and rerunning this cell changes the actual measurements.

Training/test line styles and cross-entropy/accuracy axes follow Figure 3. Automatic CE y-limits avoid misleading clipping if reproduction results have a different scale from the published paper. Line colors are selected to resemble the source figure; they are not data.

In [ ]:
# Cell 8 — PLOT-TIME DATA GENERATION from six trained checkpoint files.
# No numeric curve measurements or literal numeric constants appear in the plot function.
all_curves = OrderedDict()
all_original_endpoints = OrderedDict()
for net in CONFIG["networks"]:
    curve, original_metrics = compute_curve_directly_from_checkpoints(net)
    all_curves[net] = curve
    all_original_endpoints[net] = original_metrics


def plot_measured_figure3(curves, style, experiment_config):
    fig, axs = plt.subplots(nrows=style["rows"], ncols=len(curves),
                            figsize=style["figsize"], layout="constrained")
    for ax, (name, records) in zip(np.ravel(axs), curves.items()):
        alpha = [r["alpha"] for r in records]
        accuracy_axis = ax.twinx()
        ax.plot(alpha, [r["train_ce"] for r in records],
                color=style["loss_color"], linestyle=style["train_ls"],
                linewidth=style["line_width"], label="Train CE")
        ax.plot(alpha, [r["test_ce"] for r in records],
                color=style["loss_color"], linestyle=style["test_ls"],
                linewidth=style["line_width"], label="Test CE")
        accuracy_axis.plot(alpha, [r["train_acc"] for r in records],
                           color=style["accuracy_color"], linestyle=style["train_ls"],
                           linewidth=style["line_width"], label="Train accuracy")
        accuracy_axis.plot(alpha, [r["test_acc"] for r in records],
                           color=style["accuracy_color"], linestyle=style["test_ls"],
                           linewidth=style["line_width"], label="Test accuracy")
        ax.set_title(f"{name}: {experiment_config['models'][name]['dataset']}")
        ax.set_xlabel(r"$\alpha$ (SB at {sb}, LB at {lb})".format(
            sb=experiment_config["alpha_sb"], lb=experiment_config["alpha_lb"]))
        ax.set_ylabel("Mean cross-entropy", color=style["loss_color"])
        accuracy_axis.set_ylabel("Accuracy (%)", color=style["accuracy_color"])
        accuracy_axis.set_ylim(*style["accuracy_ylim"])
        ax.set_xlim(experiment_config["alpha_min"], experiment_config["alpha_max"])
        ax.grid(alpha=style["grid_alpha"])
        lines_left, labels_left = ax.get_legend_handles_labels()
        lines_right, labels_right = accuracy_axis.get_legend_handles_labels()
        ax.legend(lines_left + lines_right, labels_left + labels_right, loc="best")
    return fig


fig = plot_measured_figure3(all_curves, PLOT_STYLE, CONFIG)
plot_path = OUT_DIR / PLOT_STYLE["file_name"]
fig.savefig(plot_path, dpi=PLOT_STYLE["dpi"])
plt.show()
print("Figure written to", plot_path)

for net, curve in all_curves.items():
    csv_path = OUT_DIR / f"{net}_freshly_measured_curve.csv"
    with csv_path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(next(iter(curve)).keys()))
        writer.writeheader()
        writer.writerows(curve)
    print("Curve CSV (audit only; not loaded for plots):", csv_path)

with (OUT_DIR / "experiment_config.json").open("w") as stream:
    json.dump({"training_and_evaluation": CONFIG, "plot_style": PLOT_STYLE},
              stream, indent=2)

## 7. Numerical endpoint checks, caveats, and expected reference accuracies

The table below compares the **raw trained checkpoint** (using its saved BN statistics) against the **recalibrated curve endpoint** (using the fixed train-only BN subset). This difference must not be suppressed: at α=0 and α=1, the trainable parameters match their respective checkpoints, but the evaluation BN state can differ. The original paper's Table 2 test accuracies were **F1 SB/LB 98.03%/97.81%, C1 80.04%/77.26%, C2 89.24%/87.26%** (means over five runs). These numbers are **for comparison in this explanatory paragraph only** and never enter the plotted data or any stopping/selection rule.

**Conclusions you may and may not draw:** the curves reveal loss/accuracy on *one linear direction between two trained parameter vectors*, not sharpness in all directions. The paper's exact optimizer schedule, initialization, BN calibration, convolution padding, stopping tolerance and physical batch implementation are not specified; the choices above can materially change curve shapes and endpoint accuracies. A single T4 may hit Colab session limits before completion; saved progress allows resuming. Do not interpret incompletely converged endpoints as reproducing the paper's converged minima.

In [ ]:
# Cell 9 — Endpoint diagnostic printout; uses the actual results of the preceding plot cell.
for name in CONFIG["networks"]:
    endpoints = all_original_endpoints[name]
    records = all_curves[name]
    print(f"\n{name}  (orig = saved training-time BN; curve = recalibrated BN)")
    print(" regime | eval policy | train CE | test CE | train acc (%) | test acc (%)")
    for regime, target_alpha in (("SB", CONFIG["alpha_sb"]), ("LB", CONFIG["alpha_lb"])):
        orig = endpoints[regime]
        curve_point = min(records, key=lambda r: abs(r["alpha"] - target_alpha))
        assert abs(curve_point["alpha"] - target_alpha) < 1e-12
        print(f" {regime:6} | {'orig':11} | {orig['train']['ce']:8.4f} | "
              f"{orig['test']['ce']:7.4f} | {orig['train']['accuracy']:13.2f} | "
              f"{orig['test']['accuracy']:12.2f}")
        print(f" {regime:6} | {'curve':11} | {curve_point['train_ce']:8.4f} | "
              f"{curve_point['test_ce']:7.4f} | {curve_point['train_acc']:13.2f} | "
              f"{curve_point['test_acc']:12.2f}")
print("\nFinal output directory:", OUT_DIR)